
# Chapter 2 - Pre-Model Workflow and Data Preprocessing

Notebook ini membahas tahapan preprocessing data sebelum proses pelatihan model machine learning. Fokus utama chapter ini meliputi penanganan missing values, feature scaling, encoding data kategorikal, penggunaan pipeline, dan feature engineering.


## Pengaruh Kualitas Data


Data mentah sering mengandung missing value, outlier, perbedaan skala fitur, data kategorikal, hingga risiko data leakage.
Semua masalah tersebut dapat menurunkan performa model jika tidak ditangani dengan benar.


## Handling Missing Data


Metode yang dibahas:
- SimpleImputer
- KNNImputer
- IterativeImputer


In [1]:

import numpy as np
import pandas as pd

np.random.seed(42)

df = pd.DataFrame({
    "A":[10,20,np.nan,40,50],
    "B":[5,np.nan,15,20,25],
    "C":[100,120,130,np.nan,180]
})

df


,A,B,C
0,10.0,5.0,100.0
1,20.0,NaN,120.0
2,NaN,15.0,130.0
3,40.0,20.0,NaN
4,50.0,25.0,180.0



### SimpleImputer

SimpleImputer mengganti nilai kosong menggunakan statistik sederhana seperti mean, median, atau modus.


In [2]:

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

df_simple = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

df_simple


,A,B,C
0,10.0,5.00,100.0
1,20.0,16.25,120.0
2,30.0,15.00,130.0
3,40.0,20.00,132.5
4,50.0,25.00,180.0



### KNNImputer

KNNImputer memperkirakan nilai yang hilang berdasarkan tetangga terdekat (nearest neighbors).


In [3]:

from sklearn.impute import KNNImputer

knn = KNNImputer(n_neighbors=2)

df_knn = pd.DataFrame(
    knn.fit_transform(df),
    columns=df.columns
)

df_knn


,A,B,C
0,10.0,5.0,100.0
1,20.0,10.0,120.0
2,30.0,15.0,130.0
3,40.0,20.0,155.0
4,50.0,25.0,180.0



### IterativeImputer

IterativeImputer menggunakan pendekatan regresi untuk memprediksi nilai yang hilang berdasarkan fitur lainnya.


In [4]:

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iter_imp = IterativeImputer(random_state=42)

df_iter = pd.DataFrame(
    iter_imp.fit_transform(df),
    columns=df.columns
)

df_iter


,A,B,C
0,10.00000,5.000000,100.000000
1,20.00000,10.853627,120.000000
2,25.00043,15.000000,130.000000
3,40.00000,20.000000,160.000047
4,50.00000,25.000000,180.000000



## Feature Scaling

Scaling diperlukan terutama untuk algoritma berbasis jarak seperti KNN, SVM, dan beberapa metode optimisasi.



### StandardScaler

Mengubah data menjadi distribusi dengan rata-rata 0 dan standar deviasi 1.


In [5]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled = scaler.fit_transform(df_iter)

pd.DataFrame(scaled, columns=df.columns)


,A,B,C
0,-1.330274,-1.461798,-1.330266
1,-0.630133,-0.620479,-0.630126
2,-0.280032,-0.024538,-0.280056
3,0.770149,0.694092,0.770155
4,1.470291,1.412723,1.470293



### MinMaxScaler

Menskalakan fitur ke rentang tertentu, biasanya 0–1.


In [6]:

from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()

scaled_mm = minmax.fit_transform(df_iter)

pd.DataFrame(scaled_mm, columns=df.columns)


,A,B,C
0,0.000000,0.000000,0.000000
1,0.250000,0.292681,0.250000
2,0.375011,0.500000,0.375000
3,0.750000,0.750000,0.750001
4,1.000000,1.000000,1.000000



### Normalizer

Melakukan normalisasi per baris sehingga setiap sampel memiliki panjang vektor yang sama.


In [7]:

from sklearn.preprocessing import Normalizer

normalizer = Normalizer()

normalized = normalizer.fit_transform(df_iter)

pd.DataFrame(normalized, columns=df.columns)


,A,B,C
0,0.099381,0.049690,0.993808
1,0.163749,0.088863,0.982492
2,0.187650,0.112588,0.975762
3,0.240772,0.120386,0.963087
4,0.265279,0.132640,0.955005



## Encoding Categorical Variables

Sebagian besar algoritma machine learning hanya menerima input numerik sehingga data kategorikal harus dikonversi terlebih dahulu.


In [8]:

categorical_df = pd.DataFrame({
    "Department":["IT","HR","IT","Finance"],
    "Position":["Junior","Senior","Manager","Junior"]
})

categorical_df


,Department,Position
0,IT,Junior
1,HR,Senior
2,IT,Manager
3,Finance,Junior



### One Hot Encoding

Membuat kolom biner untuk setiap kategori.


In [9]:

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(categorical_df)

pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out()
)


,Department_Finance,Department_HR,Department_IT,Position_Junior,Position_Manager,Position_Senior
0,0.0,0.0,1.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0



### Label Encoding

Mengubah kategori menjadi angka integer.


In [10]:

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

temp = categorical_df.copy()

for col in temp.columns:
    temp[col] = le.fit_transform(temp[col])

temp


,Department,Position
0,2,0
1,1,2
2,2,1
3,0,0



### ColumnTransformer

Memungkinkan preprocessing berbeda untuk kolom numerik dan kategorikal secara bersamaan.


In [11]:

from sklearn.compose import ColumnTransformer

mixed = pd.DataFrame({
    "Age":[25,30,35,40],
    "Salary":[5000,7000,8000,10000],
    "Department":["IT","HR","Finance","IT"]
})

transformer = ColumnTransformer([
    ("num", StandardScaler(), ["Age","Salary"]),
    ("cat", OneHotEncoder(), ["Department"])
])

result = transformer.fit_transform(mixed)
result


array([[-1.34164079, -1.38675049,  0.        ,  0.        ,  1.        ],
       [-0.4472136 , -0.2773501 ,  0.        ,  1.        ,  0.        ],
       [ 0.4472136 ,  0.2773501 ,  1.        ,  0.        ,  0.        ],
       [ 1.34164079,  1.38675049,  0.        ,  0.        ,  1.        ]])


## Pipeline

Pipeline membantu menggabungkan preprocessing dan modeling menjadi satu alur kerja yang konsisten.


In [12]:

from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    random_state=42
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

pipe.fit(X_train, y_train)

print("Accuracy:", pipe.score(X_test, y_test))


Accuracy: 0.9790209790209791



## Mengapa Pipeline Penting?

Keuntungan utama:

- Mengurangi risiko data leakage.
- Menjamin konsistensi preprocessing.
- Mempermudah hyperparameter tuning.
- Membuat kode lebih rapi dan mudah direproduksi.



## Data Leakage

Salah satu kesalahan paling umum adalah melakukan transformasi terhadap seluruh dataset sebelum train-test split.

Praktik yang benar:

1. Split data terlebih dahulu.
2. Fit preprocessing hanya pada data training.
3. Terapkan transformasi ke data testing menggunakan parameter yang telah dipelajari dari data training.



## Feature Engineering

Feature engineering merupakan proses membuat fitur baru atau memperbaiki representasi fitur yang sudah ada agar model dapat menangkap pola dengan lebih baik.

Contoh:
- Rasio penjualan terhadap stok.
- Lama pelanggan bergabung.
- Kategori usia.
- Total transaksi per bulan.



## Ringkasan Chapter

Konsep utama yang dipelajari:

| Topik | Tujuan |
|--------|---------|
| Missing Value | Mengisi atau menangani data kosong |
| Scaling | Menyamakan skala fitur |
| Encoding | Mengubah data kategorikal menjadi numerik |
| ColumnTransformer | Mengelola berbagai tipe data |
| Pipeline | Mengotomatisasi workflow preprocessing |
| Feature Engineering | Meningkatkan kualitas representasi fitur |
| Data Leakage | Mencegah kebocoran informasi saat training |
